<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Autonomous Traders</h2>
            <span style="color:#ff7800;">An equity trading simulation to illustrate autonomous agents powered by tools and resources from MCP servers.
            </span>
        </td>
    </tr>
</table>




# Autonomous Traders

An equity trading simulation, with 4 Traders and a Researcher, powered by a slew of MCP servers with tools & resources:

1. Our home-made Accounts MCP server (written by our engineering team!)
2. Fetch (get webpage via a local headless browser)
3. Memory
4. Brave Search
5. Financial data

And a resource to read information about the trader's account, and their investment strategy.

The goal is to make a new python module, `traders.py` that will manage a single trader on our trading floor.




In [14]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

load_dotenv(override=True)

True

## Gathering the MCP params for the trader

In [15]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(is_paid_polygon)
print(is_realtime_polygon)

False
False


In [16]:
if is_paid_polygon or is_realtime_polygon:
    market_mcp = {"command": "uvx","args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@master", "mcp_polygon"], "env": {"POLYGON_API_KEY": polygon_api_key}}
else:
    market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

### And now for researcher

In [17]:
from dotenv import load_dotenv
load_dotenv(override=True)

tavily_env = {**os.environ, "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}

researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "npx", "args": ["-y", "tavily-mcp"], "env": tavily_env}
]

### Now create the MCPServerStdio for each

In [26]:
import os
os.environ["PATH"] += ":/Users/girisha/.local/bin:/usr/local/bin:/opt/homebrew/bin"

from dotenv import load_dotenv
load_dotenv(override=True)

researcher_mcp_server_params = [
    {"command": "/Users/girisha/.local/bin/uvx", "args": ["mcp-server-fetch"]},
    {"command": "/usr/local/bin/npx", "args": ["-y", "tavily-mcp"], "env": {**os.environ, "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}}
]

researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

### Make a Researcher Agent to do market research


In [27]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""You are a financial researcher. You are able to search the web for interesting financial news,
look for possible trading opportunities, and help with research.
Based on the request, you carry out necessary research and respond with your findings.
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
If there isn't a specific request, then just respond with investment opportunities based on searching latest news.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
    )
    return researcher

In [28]:
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="This tool researches online for news and opportunities, \
                either based on your specific request to look into a certain stock, \
                or generally for notable financial news and opportunities. \
                Describe what kind of research you're looking for."
        )

In [29]:
research_question = "What's the latest news on Amazon?"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))



Here's a summary of the latest news on Amazon as of today, June 11, 2026:

1. Amazon is cutting about 16,000 jobs, following a previous round of 14,000 layoffs in October.
2. Amazon is closing its Amazon Fresh grocery and Go convenience stores to scale back its brick-and-mortar grocery business.
3. Eligible Amazon customers can file refund claims as part of a $2.5 billion FTC settlement related to deceptive Prime enrollment practices.
4. There is a lawsuit against Amazon's Ring facial recognition software, alleging privacy violations.
5. Amazon's Blue Origin faced a setback with a rocket explosion during a test run in Florida.
6. Amazon is investing more than €15 billion in France, creating over 7,000 permanent jobs.
7. Amazon CEO Andy Jassy highlighted Amazon's focus on low retail prices, rural America investment, and opening its supply chain to all businesses.
8. Amazon Prime Day is scheduled from June 23 to 26, featuring new Alexa AI-powered shopping features.
9. AWS launched new EC2 instances powered by the Graviton5 processor, offering improved performance and energy efficiency.
10. Amazon Ads introduced new AI-powered tools to enhance advertising campaigns' effectiveness.

Let me know if you want more details on any specific topic or aspect!

### Look at the trace

https://platform.openai.com/traces

In [30]:
ed_initial_strategy = "You are a day trader that aggressively buys and sells shares based on news and market conditions."
Account.get("Ed").reset(ed_initial_strategy)

display(Markdown(await read_accounts_resource("Ed")))
display(Markdown(await read_strategy_resource("Ed")))

{"name": "ed", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-06-11 13:49:41", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

You are a day trader that aggressively buys and sells shares based on news and market conditions.

### And now - to create Trader Agent

In [31]:
agent_name = "Ed"

# Using MCP Servers to read resources
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
You are a trader that manages a portfolio of shares. Your name is {agent_name} and your account is under your name, {agent_name}.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
{strategy}
Your current holdings and balance is:
{account_details}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.
Please make use of these tools to manage your portfolio. Carry out trades as you see fit; do not wait for instructions or ask for confirmation.
"""

prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [32]:
print(instructions)


You are a trader that manages a portfolio of shares. Your name is Ed and your account is under your name, Ed.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
You are a day trader that aggressively buys and sells shares based on news and market conditions.
Your current holdings and balance is:
{"name": "ed", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-06-11 13:49:41", 10000.0], ["2026-06-11 13:49:44", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.
Please

### And to run our Trader

In [33]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="gpt-4o-mini",
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

### Summary of Portfolio Actions

1. **Investigation of Current Market Conditions:** 
   - Reviewed the market news, focusing on AI-driven earnings growth and geopolitical risks, particularly in the energy sector due to the Iran conflict.
   
2. **Stock Price Check:**
   - **Nvidia (NVDA):** $200.42
   - **CrowdStrike (CROWD):** Price not available; may need further investigation.
   - **Peloton (PTON):** $5.59
   - **Apple (AAPL):** $291.58
   - **Amazon (AMZN):** $238

3. **Executed Trades:**
   - **Buys:**
     - **Nvidia (NVDA):** Purchased 10 shares due to strong growth prospects in AI.
     - **Apple (AAPL):** Purchased 20 shares, as a tech leader with solid performance.
     - **CrowdStrike (CROWD):** Attempted to purchase 15 shares based on favorable earnings, but transaction failed due to an unrecognized stock symbol.
   
   - **Sells:**
     - **Peloton (PTON):** Attempted to sell 5 shares, but didn’t hold any shares.
     - **Amazon (AMZN):** Attempted to sell 5 shares, but didn’t hold any shares.

4. **Issues Encountered:**
   - Unable to execute the purchase for CrowdStrike.
   - Attempts to sell shares of Peloton and Amazon failed due to not holding any shares.

### Next Steps:
- **Further Investigation:** Check why the stocks CRWD or any other symbols could not be recognized. 
- **Adjust Positions:** Consider other stocks for potential trading opportunities based on the evolving market conditions. 
- **Monitoring:** Keep an eye on the recent purchases and adjust the portfolio as market conditions change. 

Would you like to perform any additional actions or specific investigations?

### Then go and look at the trace

http://platform.openai.com/traces


In [ ]:


await read_accounts_resource(agent_name)

'{"name": "ed", "balance": 2148.5284, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"NVDA": 10, "AAPL": 20}, "transactions": [{"symbol": "NVDA", "quantity": 10, "price": 200.82083999999998, "timestamp": "2026-06-11 13:50:12", "rationale": "Strong earnings growth driven by AI demand; positioned well for future growth."}, {"symbol": "AAPL", "quantity": 20, "price": 292.16316, "timestamp": "2026-06-11 13:50:12", "rationale": "Despite pullbacks, Apple remains a tech leader with solid fundamentals."}], "portfolio_value_time_series": [["2026-06-11 13:49:41", 10000.0], ["2026-06-11 13:49:44", 10000.0], ["2026-06-11 13:50:12", 9995.991600000001], ["2026-06-11 13:50:12", 9984.328399999999], ["2026-06-11 13:51:50", 9984.328399999999]], "total_portfolio_value": 9984.328399999999, "total_profit_loss": -15.67160000000149}'